In [1]:
import warnings
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

In [2]:
import os
import re

def find_py_files(root_dir):
    py_files = []
    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith('.py'):
                py_files.append(os.path.join(dirpath, filename))
    return py_files

def detect_and_repair_syntax_errors_in_functions(file_content):
    lines = file_content.split('\n')
    repaired_lines = []
    changes = []
    func_def_pattern = re.compile(r'^\s*def\s+(\w+)\s*\((.*)\)\s*:?')
    for i, line in enumerate(lines):
        match = func_def_pattern.match(line)
        if match:
            # Check for missing colon
            if not line.rstrip().endswith(':'):
                repaired_line = line.rstrip() + ':'
                changes.append(f"Line {i+1}: Added missing colon in function definition.")
                repaired_lines.append(repaired_line)
                continue
            # Check for unmatched parentheses
            open_paren = line.count('(')
            close_paren = line.count(')')
            if open_paren != close_paren:
                repaired_line = line + (')' * (open_paren - close_paren))
                changes.append(f"Line {i+1}: Fixed unmatched parentheses in function definition.")
                repaired_lines.append(repaired_line)
                continue
        repaired_lines.append(line)
    return '\n'.join(repaired_lines), changes

root_dir = '.'
py_files = find_py_files(root_dir)
file_changes = {}

for py_file in py_files:
    with open(py_file, 'r', encoding='utf-8') as f:
        content = f.read()
    repaired_content, changes = detect_and_repair_syntax_errors_in_functions(content)
    if changes:
        with open(py_file, 'w', encoding='utf-8') as f:
            f.write(repaired_content)
        file_changes[py_file] = changes

# Document the changes for each file
for file, changes in file_changes.items():
    print(f"Changes made to {file}:")
    for change in changes:
        print(f"  - {change}")

In [3]:
import subprocess

# Use the list of py_files from previous code
failed_files = {}

for py_file in py_files:
    try:
        # Attempt to run the file using subprocess
        result = subprocess.run(
            ["python", py_file],
            capture_output=True,
            text=True,
            timeout=30
        )
        if result.returncode != 0:
            failed_files[py_file] = result.stderr
    except Exception as e:
        failed_files[py_file] = str(e)

# Report only files that failed to execute
if failed_files:
    print("The following files failed to execute:")
    for file, error in failed_files.items():
        print(f"\n{file}:\n{error}")
else:
    print("All .py files executed successfully without errors.")

All .py files executed successfully without errors.
